# Criação das contagens

Este notebook tem como objetivo principal extrair os alarmes únicos da base e realizar a contagem de ocorrências nas últimas 6h e 12h.

---

# Import dos dados e bibliotecas

In [1]:
import pandas as pd
import numpy as np

import pyarrow

In [2]:
df_caminhao = pd.read_parquet('../datasets/datasets_tratados/df_caminhao_limpo.parquet', engine='pyarrow')

# Contagem dos erros

In [3]:
# Garantindo que os dados estejam ordenados
df_caminhao = df_caminhao.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)

# One-Hot Encoding
df_dummies = pd.get_dummies(df_caminhao['Alarme'], dtype=int)

# Ajuste do índice
df_temp = pd.concat([df_caminhao[['TAG', 'Data_Evento']], df_dummies], axis=1)
df_temp = df_temp.set_index('Data_Evento')

# Definindo parâmetros
janelas = {'6h': '6h', '12h': '12h'}
lista_resultados = []

# Iterando sobre as janelas para garantir a contagem correta
for nome, tempo in janelas.items():
    print(f"Calculando soma rolante de {nome}...")
    
    # Fazendo a contagem com base no groupby para garantir que não há vazamento de dados
    df_rolling = df_temp.groupby('TAG').rolling(tempo).sum().reset_index()

    # Ajustes de implementação
    df_rolling = df_rolling.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)
    
    novas_cols = {col: f'Contagem_{nome}_{col}' for col in df_rolling.columns if col not in ['TAG', 'Data_Evento']}
    df_rolling = df_rolling.rename(columns=novas_cols)
    
    # Guardamos apenas as colunas de contagem na lista
    lista_resultados.append(df_rolling.drop(columns=['TAG', 'Data_Evento']))

# Concatenando as colunas de contagem aos dados
df_contagens = pd.concat(lista_resultados, axis=1)
df_caminhao = pd.concat([df_caminhao, df_contagens], axis=1)

Calculando soma rolante de 6h...
Calculando soma rolante de 12h...


In [4]:
# Salva os arquivos com as contagens

df_caminhao.to_parquet('../datasets/datasets_tratados/df_caminhao_modelagem_ohe.parquet', index=False)